# 3. LDA results and comparison exports

This notebook exports topic summaries and publication results, then publishes the selected LDA
run within the separate exploration directory. Run notebooks 01 and 02 first. Selection and label confirmation
remain explicit team tasks; missing inputs stay pending.

Publication shares describe this sample. Publication size, date coverage, boilerplate, and the
keywords used to collect articles can all affect the themes we find.

This notebook retains the PR’s 100-iteration, default-prior exploration in
`Results/classical_exploration/`. The selected model from `main` uses
`selected_configuration.json` (K=40, 50 iterations, priors 0.1/0.025)
and is rerun with `classical-methods/main.py`. These are different experiments;
the exploration does not replace the selected handoff or its labels.


In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/classical-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import CORPUS_DIR, audit_corpus, load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
import lda as ld
import representation as rep
import notebook_interpret as interpretation
import notebook_artifacts as artifacts
OUT = ROOT.parent/'modern-methods/Results/classical_exploration'
MODERN = ROOT.parent/'modern-methods/Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
df = load_corpus()
reservation = artifacts.load_reservation(MODERN/'review_sample.json', df)
X, vocabulary, preparation = rep.load_representation(df, OUT/'cache/representation')
implementation = artifacts.implementation_hash(ROOT, ROOT/'notebooks/02_classical_topic_modeling.ipynb')
expected_runs = [f'lda_k{k}_s42' for k in [10,20,30,40,50,60,70]]
complete_runs = []
for run_id in expected_runs:
    folder = OUT/'runs'/run_id
    if not (folder/'manifest.json').exists():
        continue
    saved = json.loads((folder/'manifest.json').read_text())
    expected = artifacts.run_identity(df, preparation, saved['parameters'], implementation, reservation)
    _, saved = artifacts.load_run(folder, expected=expected, load_model=False)
    ev.validate_classical(folder, df.article_id.tolist())
    complete_runs.append(run_id)
selection = artifacts.selection_record(OUT, df, complete_runs)
print(f'{len(complete_runs)} of 7 grid runs complete; selection:', selection['run_id'] if selection else 'pending')

7 of 7 grid runs complete; selection: pending


## Topic shares by publication

All publications and topics appear in the CSV tables, including topics with zero dominant
articles. The main heatmap shows publications with at least 50 articles and the 15 largest
topics, plus an all-zero unassigned column for compatibility with the modern figures.

Shares use **all articles from that publication** in the denominator. Displayed columns may
sum to less than 100%. The selected model uses confirmed names where available; other names
remain keyword labels. Read article evidence before writing thematic conclusions.

In [3]:
for run_id in complete_runs:
    folder = OUT/'runs'/run_id
    inventory = interpretation.topic_inventory(folder, folder, len(df))
    assignments = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    artifacts.publication_results(df, assignments, inventory, folder/'publication')
labels_ready = False
handoff = None
if selection is not None:
    run_id = selection['run_id']
    folder, selected = OUT/'runs'/run_id, OUT/'selected'/run_id
    manifest = json.loads((folder/'manifest.json').read_text())
    identity = dict(run_id=run_id,corpus_hash=corpus_hash(df),run_identity_hash=digest(manifest['identity']))
    labels = interpretation.labels_table(folder, selected, identity)
    labels_ready = bool(len(labels) and labels.confirmed.astype(str).str.lower().eq('true').all())
    inventory = interpretation.topic_inventory(folder, selected, len(df), labels)
    assignments = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    artifacts.publication_results(df, assignments, inventory, selected/'publication')
    display(inventory)
    figure = selected/'publication/publication_proportions.png'
    if figure.exists(): display(Image(filename=str(figure)))
else:
    print('Selected-model interpretation pending team choice in selection.json.')

Selected-model interpretation pending team choice in selection.json.


## Export the exploratory handoff

The exploratory handoff stays in `modern-methods/Results/classical_exploration/`. We publish
`assignments.csv`, `terms.csv`, `topics.csv`, and `manifest.json` there, keeping full topic mixtures
in a separate local `mixtures.csv`. Every article has one dominant topic; all model topics remain
in the topic table, including zero-count topics.

The manifest records the actual settings, corpus and reservation identity, software/hardware,
metrics, stage timings, and whether all names are confirmed. Numerical comparison can use keyword
labels, but human-review packets require confirmed names in both methods.

If an earlier comparison binds different exports, publication stops. Preserve the entire prior
comparison directory under `Results/comparison/archive/` before rebuilding. Never silently replace
frozen review files or answers. Cache-load timings are not substituted for fitting times.

In [4]:
if selection is not None:
    handoff = artifacts.publish_handoff(folder, selected, OUT, df, reservation)
    ev.validate_classical(OUT, df.article_id.tolist())
    display(pd.Series(handoff['timings'],name='Original measured stage seconds'))
    print('Validated LDA handoff:', OUT)
    print('Labels confirmed:', handoff['labels_confirmed'])
else:
    print('LDA handoff pending team selection; no selected exports are fabricated.')

LDA handoff pending team selection; no selected exports are fabricated.


## Readiness and next steps

The [comparison notebook](../../comparison/pipeline_comparison.ipynb) reads the selected CLI exports
from `Results/classical/`, not this exploration. Using an exploratory model in a new comparison
requires a separately identified study and an explicit choice of its exports.
It computes shared coherence/diversity, lexical overlap, assignment contingency, publication
comparisons, and recorded runtime context. Finalize both sets of names before creating the
independent-review packets. Real reviewers supply ratings in their separate notebooks.

Discuss instability, mixed membership, different topic granularities, and BERTopic's unassigned
articles when interpreting results. Missing human scores remain missing. We update the report
separately; the historical PR2 numbers are not results from this new grid.

In [5]:
stability_ready = selection is not None and (OUT/'selected'/selection['run_id']/'stability.csv').exists()
evidence = pd.DataFrame([
    ('Shared corpus and representation','complete','representation_summary.json'),
    ('Seven-configuration 100-iteration grid','complete' if len(complete_runs)==7 else 'partial','experiments.csv; runs/*/manifest.json'),
    ('Team model choice','complete' if selection else 'pending team','selection.json'),
    ('Selected seeds 7 and 21','complete' if stability_ready else 'pending selection/rerun','selected/<run>/stability.csv'),
    ('Confirmed names and evidence','complete' if labels_ready else 'pending team','selected/<run>/topic_labels.csv'),
    ('Validated comparison exports','complete' if handoff else 'pending selection','assignments.csv; terms.csv; topics.csv; manifest.json'),
    ('Joint comparison and independent ratings','pending both methods and reviewers','../comparison; ../human-review'),
    ('Held-out and shared-date studies','deferred','Final-report follow-up')
],columns=['evidence','status','artifact'])
evidence.to_csv(OUT/'evidence_checklist.csv',index=False)
write_json(OUT/'status.json',dict(corpus_hash=corpus_hash(df), completed_grid_runs=complete_runs,
           selected_run=selection['run_id'] if selection else None, labels_confirmed=labels_ready,
           handoff_ready=handoff is not None, selected_seeds_ready=stability_ready))
display(evidence)

,evidence,status,artifact
0,Shared corpus and representation,complete,representation_summary.json
1,Seven-configuration 100-iteration grid,complete,experiments.csv; runs/*/manifest.json
2,Team model choice,pending team,selection.json
3,Selected seeds 7 and 21,pending selection/rerun,selected/<run>/stability.csv
4,Confirmed names and evidence,pending team,selected/<run>/topic_labels.csv
5,Validated comparison exports,pending selection,assignments.csv; terms.csv; topics.csv; manife...
6,Joint comparison and independent ratings,pending both methods and reviewers,../comparison; ../human-review
7,Held-out and shared-date studies,deferred,Final-report follow-up
